#  실습: 메모리와 대화 맥락



---

## 🎯 이 노트북에서 만들 것



```
1️⃣ Stateless 문제 직접 체험 (LLM은 정말 기억 못 한다)
       ↓
2️⃣ 수동 메모리 — history 리스트 직접 관리
       ↓ (사용자 1만 명이면? → 한계 명확)
3️⃣ 신 방식 4단계 — RunnableWithMessageHistory ⭐
       ↓
4️⃣ 사용자별 격리 — session_id의 마법
       ↓
5️⃣ 히스토리 들여다보기 + 초기화
       ↓
6️⃣ 토큰 폭발 방지 — trim_messages
       ↓
🎁 도전과제: 페르소나 + 메모리 챗봇 · 같은 유저의 페르소나별 분리
```

## 📖 이론 매핑

| 이론 문서 섹션 | 이 노트북에서 다루는 코드 |
|---|---|
| 1부. Stateless의 의미 | Step 1. 이름 기억 못 함 시연 |
| 들어가며. 수동 메모리의 4가지 한계 | Step 2. 직접 만들어보고 한계 체험 |
| 4부. 신 방식 4가지 구성 요소 | Step 3. 4단계로 하나씩 조립 |
| 4부. session_id 자동 격리 | Step 4. 두 사용자 동시 사용 |
| 4부. 동작 흐름 그림 | Step 5. store 직접 출력 |
| 6부. trim_messages | Step 6. 토큰 자동 압축 |
| 4부 + Q5. 페르소나×사용자 조합 | 도전과제 1·2 |

---

## 💡 외워둘 한 줄

> ```python
> chain_with_memory.invoke(
>     {"question": "..."},
>     config={"configurable": {"session_id": "user_001"}}
> )
> ```
>
> 이 패턴이 4-5교시 챗봇의 심장이에요.


---

# 🛠 Step 0. 환경 준비




In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5-nano', temperature = 0.3)
print('LLM 준비 완료')

LLM 준비 완료


---

# 1️⃣ Step 1. LLM은 정말 대화를 기억 못 한다

> **이론 매핑**: 1부. Stateless의 의미

직접 두 번 호출해서 LLM이 첫 번째 호출을 전혀 기억하지 못함을 확인합시다.


In [6]:
# 1차 호출 — 이름 알려주기
result1 = llm.invoke("내 이름은 다빈이야. 잘 부탁해.")
print("[1차]", result1.content)
print("-"*50)

# 2차 호출 — 이름 다시 물어보기 (별개의 호출!)
result2 = llm.invoke("내 이름이 뭐였지?")
print("[2차]", result2.content)


[1차] 다빈님, 반갑습니다! 앞으로 잘 부탁드립니다. 어떤 도움이 필요하신지 알려주시면 최선을 다해 돕겠습니다.

필요한 것 예시
- 글쓰기 첨삭/교정
- 영어/한국어 번역
- 공부 계획이나 일정 정리
- 아이디어 발상, 브레인스토밍
- 코딩 문제 풀이
- 여행 계획이나 레시피 찾기
- 대화 연습 등

원하시는 톤이나 선호하는 주제가 있나요? 오늘 바로 시작해볼까요?
--------------------------------------------------
[2차] 지금 이 대화에서 당신의 이름을 알 수 없어요. 이름을 알려주시면 이 대화 동안 그 이름으로 불러드릴게요. 당신의 이름이 뭔가요? 또한 원하시면 별명으로 불러드릴 수도 있습니다.


✅ **결과 확인**: 2차 답변에 "다빈"이 안 나옵니다.  
LLM 입장에서 두 호출은 **완전히 별개**예요. 매번 백지 상태에서 시작합니다.

> 💡 ChatGPT 사이트가 기억하는 것처럼 보이는 이유는, 이전 대화를 **매번 함께 보내주는** 시스템이 뒤에 있기 때문이에요. 그걸 우리도 구현해야 합니다.


---

# 2️⃣ Step 2. 수동 메모리 — 직접 해결해보기

가장 직관적인 방법: **메시지 리스트를 우리가 직접 관리**해서 매번 함께 보내기.


In [7]:
# 메시지 리스트를 직접 들고 다님
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = [SystemMessage("당신은 친절한 AI 어시스턴트입니다.")]

def manual_chat(user_message: str) -> str:
    """리스트에 메시지를 직접 누적해서 LLM 호출"""
    history.append(HumanMessage(user_message))   # 사용자 메시지 추가
    result = llm.invoke(history)                  # 전체 히스토리로 호출
    history.append(AIMessage(result.content))     # AI 답변도 누적
    return result.content


# 1차
print("[1차]", manual_chat("내 이름은 다빈이야."))
print("-"*50)
# 2차 — 이번엔 이전 대화가 함께 전송됨
print("[2차]", manual_chat("내 이름이 뭐였지?"))


[1차] 다빈님, 반가워요! 이 대화에서 다빈님이라고 부르겠습니다. 오늘 어떤 도움을 원하시나요? 예를 들면:
- 필요한 정보 검색이나 요약
- 글쓰기나 아이디어 브레인스토밍
- 공부나 언어 연습
- 일정 관리나 계획 세우기
- 기술 문제 해결

원하시는 방향을 말씀해 주시면 바로 도와드릴게요.
--------------------------------------------------
[2차] 다빈이에요. 필요하시면 다빈님으로 불러드릴게요.


✅ **결과 확인**: 이번엔 "다빈"이라고 답변하죠? **수동 메모리도 작동은 합니다.**

### 그런데 문제가 4가지 있어요

| 문제 | 무엇이 곤란한가 |
|---|---|
| **사용자 1만 명** | `history` 리스트를 사용자마다 따로 관리해야 함 |
| **대화 100턴** | 100턴치를 매번 보내면 토큰 비용 폭증 |
| **LCEL 체인과 결합** | `prompt \| llm` 우아한 방식에 끼우기 애매 |
| **저장소 교체** | 메모리 → Redis 바꾸려면 코드 전체 손봐야 함 |

> 이 4가지를 한 방에 풀어주는 게 다음 Step의 **`RunnableWithMessageHistory`**입니다.


---

# 3️⃣ Step 3. 신 방식 4단계 — `RunnableWithMessageHistory` ⭐

> **이론 매핑**: 4부. 핵심 구성 요소

LangChain의 자동 메모리는 **4가지 부품**을 조립해서 만듭니다.  
하나씩 차근차근 만들어볼게요.

```
부품 1: 프롬프트에 MessagesPlaceholder 자리 만들기
부품 2: 세션별 히스토리 저장소 함수
부품 3: 기본 체인을 메모리로 감싸기
부품 4: session_id와 함께 호출
```


### 🧩 부품 1. 프롬프트에 `MessagesPlaceholder` 끼우기

이전 대화 메시지들이 들어갈 **빈자리**를 미리 만들어둡니다.


In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답하세요."),
    MessagesPlaceholder("history"),  # ← 이전 대화 메시지들이 자동으로 여기에 들어감
    ("human", "{question}"),          # ← 새 사용자 입력
])


### 🧩 부품 2. 세션별 히스토리 저장소

각 사용자(session)의 대화를 저장하는 공간을 만듭니다.  
**`session_id`를 키로 쓰는 사전(dict)** 이 핵심이에요.


In [9]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 모든 사용자의 히스토리를 담을 사전
store = {}

def get_session_history(session_id: str):
    """session_id에 해당하는 히스토리를 가져오거나 새로 만들기"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 처음 보면 빈 히스토리 생성
    return store[session_id]


### 🧩 부품 3. 체인을 메모리로 감싸기

기본 체인(`prompt | llm`)을 `RunnableWithMessageHistory`로 한 번 감싸면 끝.


In [10]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# 기본 체인
chain = prompt | llm

# 메모리 wrapper로 감싸기
chain_with_memory = RunnableWithMessageHistory(
    chain,                              # 감쌀 체인
    get_session_history,                # 히스토리 가져올 함수
    input_messages_key="question",      # 사용자 입력 변수 이름 ({question})
    history_messages_key="history",     # MessagesPlaceholder 이름
)


c:\lab\llm-workspace\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 🧩 부품 4. `session_id`와 함께 호출

이제 호출할 때 `config`로 `session_id`만 같이 넘기면 자동으로 대화가 이어집니다.


In [ ]:
# 같은 session_id로 두 번 연속 호출 — 자동으로 기억함
config = {"configurable": {"session_id": "user_001"}}

# 1차
result1 = chain_with_memory.invoke({"question": "내 이름은 다빈이야."}, config=config)
print("[1차]", result1.content)
print("-"*50)

# 2차 — 우리가 history 관리 안 했는데 LLM이 기억함!
result2 = chain_with_memory.invoke({"question": "내 이름이 뭐였지?"}, config=config)
print("[2차]", result2.content)

[1차] 알겠습니다, 다빈님! 반갑습니다. 어떻게 도와드릴까요?
--------------------------------------------------
[2차] 다빈님이십니다! 다른 질문이나 도움이 필요하시면 언제든지 말씀해 주세요.


✅ **결과 확인**: 2차 답변에 "진환"이 나왔나요? **수동 메모리 없이도 자동으로 기억합니다.**

> 🎯 **핵심**: 우리가 `history.append()`를 한 번도 안 썼는데도 메모리가 동작했어요.  
> 사용자 메시지·AI 답변 추가, 히스토리 끼워넣기 — 전부 `RunnableWithMessageHistory`가 자동으로 해줍니다.


---

### 🐛 잠깐, 이상한 코드 하나 — 빠진 무언가

위에서 `chain_with_memory.invoke()`를 부를 때 `config={...}`를 같이 넘겼던 것, 기억나죠?  
그걸 깜빡하고 안 넘기면 어떻게 될까요? **에러가 어떻게 나는지** 직접 봐두세요.

> 💡 한 번 겪어두면 평생 같은 실수 안 합니다. 정답은 다음 Step 4에서 자연스럽게 풀려요.

In [13]:
# 🐛 이 코드는 에러가 납니다. 어디가 빠졌을까요?
result = chain_with_memory.invoke({"question": "내 이름은 다빈이야."})
print(result.content)

ValueError: Missing keys ['session_id'] in config['configurable'] Expected keys are ['session_id'].When using via .invoke() or .stream(), pass in a config; e.g., chain.invoke({'question': 'foo'}, {'configurable': {'session_id': '[your-value-here]'}})

🤔 **관찰 포인트**:
- 어떤 에러 메시지가 나왔나요? `Missing keys`라는 표현이 보이나요?
- 어떤 *키*가 필요하다고 나오나요?
- 그 키를 *어디에* 넘겨줘야 할까요? (방금 Step 3 마지막 호출 코드를 다시 보세요)

> 💡 **교훈**: `RunnableWithMessageHistory`는 *반드시* `config={"configurable": {"session_id": ...}}`가 필요합니다.  
> - Day 1 Lesson 1의 분류기 버그: *조용한* 오작동 (temperature 잘못)
> - Day 1 Lesson 2의 템플릿 버그: *시끄러운* `KeyError` (변수 이름 불일치)
> - 오늘의 버그: *시끄러운* `ValueError` (config 자체가 빠짐)
>
> 에러 메시지를 천천히 읽는 습관이 LangChain 학습의 절반입니다.

다음 Step에서 이 `session_id`가 왜 그렇게 중요한지 풀립니다.

---

# 4️⃣ Step 4. `session_id`로 사용자별 격리

> **이론 매핑**: 4부. "다른 사용자는 다른 session_id"

`session_id`가 다르면 **완전히 별개의 대화**로 인식됩니다.  
사용자 A와 사용자 B를 동시에 시뮬레이션해봅시다.


In [ ]:
# 사용자 A: user_001 → "다빈"이라고 자기소개
config_a = {"configurable": {"session_id": "user_001"}}
chain_with_memory.invoke({"question": "내 이름은 다빈이야."}, config=config_a)

# 사용자 B: user_002 → "윤미"라고 자기소개
config_b = {"configurable": {"session_id": "user_002"}}
chain_with_memory.invoke({"question": "내 이름은 윤미야."}, config=config_b)

# 🎯 미션: 사용자 C 추가 — 본인 실제 이름으로 자기소개
#         session_id는 "user_003", 이름은 본인 이름을 넣어보세요.
config_c = {"configurable": {"session_id": "user_003"}}
chain_with_memory.invoke({"question": "내 이름은 ____이야."}, config=config_c)  # ← 본인 이름

# 세 사용자 모두에게 이름 다시 물어보기
ans_a = chain_with_memory.invoke({"question": "내 이름이 뭐였지?"}, config=config_a)
ans_b = chain_with_memory.invoke({"question": "내 이름이 뭐였지?"}, config=config_b)
ans_c = chain_with_memory.invoke({"question": "내 이름이 뭐였지?"}, config=config_c)

print("👤 사용자 A:", ans_a.content)
print("👤 사용자 B:", ans_b.content)
print("👤 사용자 C (본인):", ans_c.content)

👤 사용자 A: 다빈님이십니다! 다른 질문이나 도움이 필요하시면 언제든지 말씀해 주세요.
👤 사용자 B: 윤미님이시죠! 다른 질문이 있으신가요?
👤 사용자 C (본인): 죄송하지만, 당신의 이름을 알 수 있는 방법이 없어요. 이름을 다시 말씀해 주시면 좋겠습니다!


✅ **결과 확인**: A에겐 "진환", B에겐 "민지", C에겐 *본인이 입력한 이름*이 돌아와야 정상입니다.  
세 대화가 섞이지 않고 깔끔하게 분리됐어요.

> 💡 실서비스에서는 보통 로그인한 사용자 ID를 `session_id`로 그대로 씁니다. `f"user_{user.id}"`처럼요.

### 🔍 한 발 더

본인이 추가한 user_003의 응답이 user_001의 "진환"으로 *잘못* 나왔다면 어떤 종류의 버그일까요?  
(힌트: 방금 본 🐛 버그 셀과 비슷한 종류 — config 키 하나만 어긋나도 세션이 통째로 섞입니다)

---

# 5️⃣ Step 5. 저장소 들여다보기 + 초기화

> **이론 매핑**: 4부. 동작 흐름 / Q4. 초기화

`store`는 그냥 Python 사전입니다. 직접 열어볼 수 있어요. **디버깅에 매우 유용**합니다.


### 🤔 실행 전에 1분 멈춤

지금까지는 `chain_with_memory`만 호출했지, *내부의 `store`*를 직접 본 적은 없습니다.  
아래 셀을 실행하기 *전에* 예측해보세요:

- `store`의 타입은? **dict? list? 다른 객체?**
- `store["user_001"]`은 어떤 형태? 그냥 메시지 리스트? 객체?
- 그 안에 들어있는 *개별 메시지*는 어떤 형태? 문자열? 딕셔너리? LangChain 객체?

예측을 마음에 적은 뒤 실행해서 비교하세요.

In [21]:
# 📦 예측 확인: store의 구조 직접 들여다보기
print("store의 타입:", type(store).__name__)
print("저장된 세션:", list(store.keys()))
print()

# user_001 히스토리 객체 자체의 정체
history_obj = store["user_001"]
print("store['user_001']의 타입:", type(history_obj).__name__)
print()

# 사용자 A의 대화 내역 전체 출력
print("=== user_001 히스토리 ===")
for msg in history_obj.messages:
    role = type(msg).__name__.replace("Message", "")
    print(f"  [{role}] {msg.content}")

store의 타입: dict
저장된 세션: ['user_001', 'user_002', 'user_003']

store['user_001']의 타입: InMemoryChatMessageHistory

=== user_001 히스토리 ===
  [Human] 내 이름은 진환이야.
  [AI] 안녕하세요, 진환님! 만나서 반갑습니다. 어떻게 도와드릴까요?
  [Human] 내 이름이 뭐였지?
  [AI] 진환님이시죠! 다른 질문이나 도움이 필요하시면 언제든지 말씀해 주세요.
  [Human] 내 이름은 진환이야.
  [AI] 네, 진환님! 다시 말씀해 주셔서 감사합니다. 어떤 이야기를 나누고 싶으신가요?
  [Human] 내 이름이 뭐였지?
  [AI] 진환님이십니다! 다른 질문이 있으시면 언제든지 말씀해 주세요.


### 🧹 특정 세션만 초기화

사용자가 "대화 새로 시작" 버튼을 눌렀을 때 쓰는 패턴입니다.


In [22]:
# user_001만 초기화 (user_002는 유지)
store["user_001"].clear()

# 초기화 후 같은 질문
result = chain_with_memory.invoke(
    {"question": "내 이름이 뭐였지?"},
    config={"configurable": {"session_id": "user_001"}},
)
print("[초기화 후 user_001]", result.content)  # 이름 모름

# user_002는 그대로 기억하고 있어야 함
result_b = chain_with_memory.invoke(
    {"question": "내 이름이 뭐였지?"},
    config={"configurable": {"session_id": "user_002"}},
)
print("[user_002 유지]", result_b.content)  # 민지


[초기화 후 user_001] 죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 이름을 알려주시면 기억해두겠습니다!
[user_002 유지] 민지님이십니다! 다른 질문이 있으시면 언제든지 말씀해 주세요.


✅ **결과 확인**: user_001은 이름을 잊었고, user_002는 그대로 기억하고 있나요?

> 💡 5교시 Gradio에서 "대화 초기화" 버튼을 만들 때 정확히 이 코드를 쓰게 됩니다.


---

# 6️⃣ Step 6. 토큰 폭발 방지 — `trim_messages`

> **이론 매핑**: 6부. 토큰 비용 폭발 방지

대화가 100턴을 넘어가면 매 호출마다 100턴치를 보내게 됩니다. 비용도 한도도 위험해요.  
`trim_messages`가 **오래된 메시지를 자동으로 잘라줍니다**.

먼저 가짜로 긴 대화를 만들고, 자르는 효과를 눈으로 확인해봅시다.


In [23]:
from langchain_core.messages import trim_messages

# 가짜로 긴 대화 만들기 (시스템 + 20턴)
fake_history = [SystemMessage("당신은 친절한 AI입니다.")]
for i in range(20):
    fake_history.append(HumanMessage(f"질문 {i}번입니다. 좀 길게 적어볼게요 어쩌고저쩌고."))
    fake_history.append(AIMessage(f"답변 {i}번입니다. 자세히 설명드리자면 어쩌고저쩌고."))

print(f"원본 메시지 수: {len(fake_history)}개")


원본 메시지 수: 41개


### 🎲 예측 게임 — trim 후 몇 개나 남을까

방금 만든 `fake_history`는 **41개** 메시지(시스템 1 + 휴먼 20 + AI 20)입니다.  
`max_tokens=300`으로 자르면 결과가 어떻게 나올지 *실행 전에* 예측해보세요:

| 결과 | 예상 |
|---|:---:|
| 남은 메시지 수 (대략) | ___ 개 |
| 시스템 메시지가 *살아남는가* | ☐ 예 / ☐ 아니오 |
| 가장 *최근* 대화가 살아남는가 | ☐ 예 / ☐ 아니오 |
| 가장 *오래된* 대화가 살아남는가 | ☐ 예 / ☐ 아니오 |

예측을 마음에 적은 뒤 실행해서 비교하세요.  
잘 안 맞으면 `max_tokens` 값을 100, 1000으로 바꿔서 다시 돌려보면 감이 옵니다.

In [24]:
# trim_messages — 최근 메시지 위주로 토큰 한도 안에 유지
trimmer = trim_messages(
    max_tokens=300,        # 300 토큰 안에 맞추기
    strategy="last",       # 최근 것부터 살림
    token_counter=llm,     # 토큰 카운터로 LLM 사용
    include_system=True,   # 시스템 메시지는 무조건 유지
    start_on="human",      # human 메시지부터 시작 (대화 짝 유지)
)

trimmed = trimmer.invoke(fake_history)
print(f"잘린 후 메시지 수: {len(trimmed)}개")
print("\n=== 남은 메시지 ===")
for msg in trimmed:
    role = type(msg).__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:40]}...")


잘린 후 메시지 수: 11개

=== 남은 메시지 ===
  [System] 당신은 친절한 AI입니다....
  [Human] 질문 15번입니다. 좀 길게 적어볼게요 어쩌고저쩌고....
  [AI] 답변 15번입니다. 자세히 설명드리자면 어쩌고저쩌고....
  [Human] 질문 16번입니다. 좀 길게 적어볼게요 어쩌고저쩌고....
  [AI] 답변 16번입니다. 자세히 설명드리자면 어쩌고저쩌고....
  [Human] 질문 17번입니다. 좀 길게 적어볼게요 어쩌고저쩌고....
  [AI] 답변 17번입니다. 자세히 설명드리자면 어쩌고저쩌고....
  [Human] 질문 18번입니다. 좀 길게 적어볼게요 어쩌고저쩌고....
  [AI] 답변 18번입니다. 자세히 설명드리자면 어쩌고저쩌고....
  [Human] 질문 19번입니다. 좀 길게 적어볼게요 어쩌고저쩌고....
  [AI] 답변 19번입니다. 자세히 설명드리자면 어쩌고저쩌고....


✅ **결과 확인**: 41개 → 몇 개 안 되는 메시지로 줄어들었나요? **시스템 메시지는 살아있고, 최근 대화 위주로 남았어야 합니다.**

> 💡 실제 체인에 통합하려면 history가 prompt로 들어가기 전에 trimmer를 거치도록 끼우면 됩니다.  
> 처음 배울 땐 Buffer만 써도 충분하니, "이런 도구가 있다"만 기억하세요.


---

# 📝 정리: 이번 실습에서 만든 것

- [x] LLM이 stateless임을 코드로 확인
- [x] 수동 메모리(list)로 해결 → 한계 4가지 체험
- [x] **`RunnableWithMessageHistory` 4단계 조립** (이론의 핵심!)
- [x] `session_id`로 사용자별 자동 격리
- [x] `store` 직접 들여다보고 특정 세션 초기화
- [x] `trim_messages`로 토큰 자동 압축

### 🔑  핵심 패턴

```python
# 1) 프롬프트에 history 자리 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

# 2) 세션 저장소
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3) 체인을 메모리로 감싸기
chain_with_memory = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

# 4) session_id와 함께 호출
chain_with_memory.invoke(
    {"question": "..."},
    config={"configurable": {"session_id": "user_001"}},
)
```




---

# 🎁 도전과제

## 도전과제 1. 페르소나 + 메모리 챗봇

2교시에서 만든 페르소나 챗봇에 **메모리**를 결합하세요.  
"내 이름은 X" → "내 이름이 뭐였지?"가 페르소나 톤으로 자연스럽게 이어져야 합니다.

**힌트**:
- 프롬프트 system 메시지에 `{persona}` 변수 자리 만들기
- `MessagesPlaceholder("history")` 끼우기
- `RunnableWithMessageHistory`로 감싸기


In [25]:
# 📌 페르소나 + 메모리 챗봇 — 빈칸을 채우세요
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 👇 빈칸 1: 프롬프트 — system에 {persona}, history 자리, human에 {question}
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나의 말투와 태도를 일관되게 유지하세요."),
    MessagesPlaceholder(variable_name="history"),   # ← 대화 기록이 들어갈 자리
    ("human", "{question}"),
])

# 세션 저장소 (재사용)
persona_store = {}
def get_persona_history(session_id):
    if session_id not in persona_store:
        persona_store[session_id] = InMemoryChatMessageHistory()
    return persona_store[session_id]

# 👇 빈칸 2: 체인을 메모리로 감싸기
persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",   # ← 매번 새로 들어오는 사용자 입력 키
    history_messages_key="history",  # ← 위 MessagesPlaceholder와 같은 이름!
)


def chat_with_persona(session_id: str, persona: str, question: str) -> str:
    """페르소나·세션·질문을 받아 답변 반환"""
    result = persona_chain.invoke(
        {"persona": persona, "question": question},
        config={"configurable": {"session_id": session_id}},
    )
    return result.content


# 테스트 — 같은 세션에서 3턴 대화
sid = "test_user"
persona = "냉정하지만 친절한 시니어 데이터 분석가"
print("[1턴]", chat_with_persona(sid, persona, "안녕하세요. 저는 진환이라고 합니다."))
print("[2턴]", chat_with_persona(sid, persona, "표본 분산과 모분산의 차이를 한 줄로 알려주세요."))
print("[3턴]", chat_with_persona(sid, persona, "제가 누구라고 했었죠?"))


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


[1턴] 안녕하세요, 진환님. 만나서 반갑습니다. 어떤 도움을 드릴 수 있을까요?
[2턴] 표본 분산은 표본 데이터의 분산을 나타내고, 모분산은 전체 모집단의 분산을 나타내며, 표본 분산은 모집단 분산의 추정값으로 사용됩니다.
[3턴] 진환님이라고 하셨습니다. 추가로 궁금한 점이 있으신가요?


In [26]:
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나의 말투와 태도를 일관되게 유지하세요."),
    MessagesPlaceholder(variable_name="history"),   # ← 대화 기록이 들어갈 자리
    ("human", "{question}"),
])

persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",   # ← 매번 새로 들어오는 사용자 입력 키
    history_messages_key="history",  # ← 위 MessagesPlaceholder와 같은 이름!
)

✅ **결과 확인**: 3턴 답변에 "민영"이 나오면서 페르소나 톤이 유지되어야 합니다.

---

## 도전과제 2. session_id 전략 비교 — 어느 게 맞을까?

> **이론 매핑**: Q5. "같은 사용자가 다른 페르소나로 대화하면?"

같은 사람(`user_42`)이 **분석가**와 **마케터** 두 페르소나로 따로따로 대화한다고 해봅시다.  
세 가지 session_id 설계 전략 중 *어느 게 옳은가* 판단해보세요.

- **A)** `session_id = user_id` — 페르소나는 무시
- **B)** `session_id = f"{user_id}_{persona}"` — 사용자 × 페르소나 조합 ⭐ 정답일까?
- **C)** `session_id = persona` — 사용자는 무시

세 전략을 *같은 시나리오*에 흘려보내서, 어느 경우에 *대화가 섞이는지* 직접 확인하세요.

In [27]:
# 🎯 3가지 session_id 전략 — 같은 시나리오에 흘려서 결과 비교

def run_strategy(make_sid):
    """주어진 session_id 전략으로 시나리오 실행 후 마지막 답변 반환"""
    # 매 전략마다 새 store + 새 chain (전략 간 격리)
    local_store = {}
    def local_history(sid):
        if sid not in local_store:
            local_store[sid] = InMemoryChatMessageHistory()
        return local_store[sid]

    local_chain = RunnableWithMessageHistory(
        persona_prompt | llm,
        local_history,
        input_messages_key="question",
        history_messages_key="history",
    )

    USER = "user_42"

    # 1️⃣ 분석가 페르소나로 KPI 얘기
    local_chain.invoke(
        {"persona": "꼼꼼한 데이터 분석가", "question": "저는 KPI 대시보드 만드는 일을 합니다."},
        config={"configurable": {"session_id": make_sid(USER, "analyst")}},
    )

    # 2️⃣ 마케터 페르소나로 캠페인 얘기 (다른 페르소나)
    local_chain.invoke(
        {"persona": "톡톡 튀는 마케터", "question": "저는 신제품 캠페인을 준비하고 있어요."},
        config={"configurable": {"session_id": make_sid(USER, "marketer")}},
    )

    # 3️⃣ 다시 분석가 페르소나로 "내가 뭘 한다고 했지?" — 분석가 얘기만 기억해야 정상
    answer = local_chain.invoke(
        {"persona": "꼼꼼한 데이터 분석가", "question": "제가 뭘 한다고 했었죠?"},
        config={"configurable": {"session_id": make_sid(USER, "analyst")}},
    )
    return answer.content


strategies = {
    "A (user_id만 사용)":     lambda uid, persona: uid,
    "B (user_id + persona)":  lambda uid, persona: f"{uid}_{persona}",
    "C (persona만 사용)":     lambda uid, persona: persona,
}

for name, make_sid in strategies.items():
    print(f"\n{'='*60}")
    print(f"🧪 전략: {name}")
    print('='*60)
    print(run_strategy(make_sid))


🧪 전략: A (user_id만 사용)


/tmp/ipykernel_9691/3342924622.py:51: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  print(run_strategy(make_sid))


당신은 신제품 캠페인을 준비하고 있다고 말씀하셨습니다. 신제품을 시장에 성공적으로 런칭하기 위해 여러 가지 요소를 고려하고 계신 것 같아요. 추가적인 질문이나 구체적인 도움이 필요하시면 언제든지 말씀해 주세요!

🧪 전략: B (user_id + persona)
당신은 KPI 대시보드를 만드는 일을 하고 있다고 말씀하셨습니다. KPI 대시보드는 조직의 성과를 모니터링하고 분석하는 데 중요한 역할을 합니다. 이와 관련하여 더 구체적인 질문이나 논의하고 싶은 주제가 있으신가요?

🧪 전략: C (persona만 사용)
당신은 KPI 대시보드를 만드는 일을 하고 있다고 말씀하셨습니다. KPI 대시보드는 성과 지표를 시각적으로 표현하여 데이터 기반의 의사결정을 지원하는 중요한 도구입니다. 이와 관련하여 추가적인 질문이나 도움이 필요하시면 언제든지 말씀해 주세요.


### 🤔 판단해보기

세 전략의 *분석가 페르소나 답변*을 비교한 뒤 2-3줄로 답해보세요:

1. 분석가가 *KPI 대시보드*만 기억하게 한 전략은 어느 것인가? (A / B / C)
2. *마케터 캠페인 얘기*가 분석가 답변에 섞여 들어간 전략은? 왜 그렇게 됐을까?
3. C 전략(persona만)은 *user_42 말고 user_99*도 동시에 쓰면 무슨 일이 일어날까?

> 💡 **이 연습의 진짜 의미**: session_id 설계는 *대화 격리의 단위*를 정하는 일입니다.  
> 잘못 설계하면 **대화 누설(privacy leak)** 이나 맥락 혼선이 발생해요.  
> 실서비스에서 가장 흔하면서도 사고가 자주 나는 지점이라, 손으로 한 번 *섞이는 걸* 봐두는 게 중요합니다.